In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import pandas as pd
import numpy as np

In [13]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [33]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl


df_clean = df[['ISSUE','Network ID', 'Network Name', 'Native ID', 'Station ID', 'Unique Names', 'Unique Location (String)'
]].reset_index(drop=True)

mask = (
    df["ISSUE"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.startswith("delete")
)

df_kept = df[~mask]


df_network = df_kept[df_clean['Network Name'].str.contains(r".+->.+", na=False)][['Network Name', 'Station ID', 'Network ID', 'Unique Names',]]



df_network_unique_pair = df_network['Network Name'].unique()

split_ids = df_network['Network Name'].astype(str).str.split('->', expand=True)

df_network['old_name'] = split_ids[0].str.strip()
df_network['new_name'] = split_ids[1].str.strip()

df_old_net_name = df_network['old_name'].unique()
df_new_net_name = df_network['new_name'].unique()


/tmp/ipykernel_25440/2073936921.py:19: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_network = df_kept[df_clean['Network Name'].str.contains(r".+->.+", na=False)][['Network Name', 'Station ID', 'Network ID', 'Unique Names',]]


In [34]:
df_network_unique_pair

array(['BC-Air -> BCH-SiteC', 'BC-Air -> MVRD-AIR',
       'BC-Snow -> BCH-GSO-HMP', 'BC-Air -> MVRD', 'BC-FWx -> PC-FWx',
       'BC-FWx -> MVRD-WS', 'BC-Air -> PRPA', 'BC-FWx -> BC-FERN',
       'BCH-GSO-HMP -> ENV ASP'], dtype=object)

In [49]:
df_network

wanted_issues = [
'BC-FWx -> MVRD-WS',
'BC-Snow -> BCH-GSO-HMP',
'BC-FWx -> BC-FERN',
'BCH-GSO-HMP -> ENV ASP',

]

df_network_old_to_old = df_network[
    df_network["Network Name"].str.strip().isin(wanted_issues) 
]

df_network_old_to_old

,Network Name,Station ID,Network ID,Unique Names,old_name,new_name
233,BC-Snow -> BCH-GSO-HMP,2569,14.0,Aiken Lake,BC-Snow,BCH-GSO-HMP
234,BC-Snow -> BCH-GSO-HMP,2545,14.0,Barnes Creek,BC-Snow,BCH-GSO-HMP
235,BC-Snow -> BCH-GSO-HMP,2532,14.0,Disappointment Lake,BC-Snow,BCH-GSO-HMP
236,BC-Snow -> BCH-GSO-HMP,2551,14.0,East Creek,BC-Snow,BCH-GSO-HMP
237,BC-Snow -> BCH-GSO-HMP,2521,14.0,Green Mountain,BC-Snow,BCH-GSO-HMP
238,BC-Snow -> BCH-GSO-HMP,2567,14.0,Kwadacha River,BC-Snow,BCH-GSO-HMP
239,BC-Snow -> BCH-GSO-HMP,2522,14.0,Mission Ridge,BC-Snow,BCH-GSO-HMP
240,BC-Snow -> BCH-GSO-HMP,2548,14.0,Morrissey Ridge,BC-Snow,BCH-GSO-HMP
241,BC-Snow -> BCH-GSO-HMP,2547,14.0,Morrissey Ridge,BC-Snow,BCH-GSO-HMP
242,BC-Snow -> BCH-GSO-HMP,2541,14.0,Mount Revelstoke,BC-Snow,BCH-GSO-HMP


In [50]:
df_old_net_name

array(['BC-Air', 'BC-Snow', 'BC-FWx', 'BCH-GSO-HMP'], dtype=object)

In [55]:
# net_names = df_old_net_name[2:3].tolist()
net_names = ['BC-Snow']

query = text("""
SELECT v.vars_id,
       v.network_id,
       n.network_name,
       v.standard_name,
       v.net_var_name,
       v.display_name,
       v.short_name,
       v.long_description
FROM meta_network n
JOIN meta_vars v
  ON v.network_id = n.network_id
WHERE n.network_name IN :net_names
ORDER BY v.network_id;
""")

df = pd.read_sql(query, engine, params={"net_names": tuple(net_names)})
df

,vars_id,network_id,network_name,standard_name,net_var_name,display_name,short_name,long_description
0,511,14,BC-Snow,air_temperature,MIN_TEMP,Temperature (Min.),air_temperature_minimum,Minimum daily temperature
1,516,14,BC-Snow,surface_snow_thickness,SNOW_ON_THE_GROUND,Surface Snow Depth (Point),surface_snow_thickness_point,Amount of snow on the ground
2,515,14,BC-Snow,thickness_of_snowfall_amount,ONE_DAY_SNOW,Snowfall Amount,thickness_of_snowfall_amount_sum,Daily snow accumulation
3,513,14,BC-Snow,lwe_thickness_of_precipitation_amount,ONE_DAY_PRECIPITATION,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,Daily precipitation
4,512,14,BC-Snow,air_temperature,MAX_TEMP,Temperature (Max.),air_temperature_maximum,Maximum daily temperature
5,514,14,BC-Snow,thickness_of_rainfall_amount,ONE_DAY_RAIN,Rainfall Amount,thickness_of_rainfall_amount_sum,Daily rainfall
6,600,14,BC-Snow,lwe_thickness_of_precipitation_amount,Precip_Climatology,Precipitation Climatology,lwe_thickness_of_precipitation_amountt: sum wi...,Climatological mean of monthly total precipita...
7,599,14,BC-Snow,air_temperature,T_mean_Climatology,Temperature Climatology (Mean),air_temperaturet: mean within days t: mean wit...,Climatological mean of monthly mean of mean da...
8,598,14,BC-Snow,air_temperature,Tn_Climatology,Temperature Climatology (Min.),air_temperaturet: minimum within days t: mean ...,Climatological mean of monthly mean minimum da...
9,597,14,BC-Snow,air_temperature,Tx_Climatology,Temperature Climatology (Max.),air_temperaturet: maximum within days t: mean ...,Climatological mean of monthly mean maximum da...


In [54]:
# net_names = df_old_net_name[2:3].tolist()
net_names = ['BCH-GSO-HMP']

query = text("""
SELECT v.vars_id,
       v.network_id,
       n.network_name,
       v.standard_name,
       v.net_var_name,
       v.display_name,
       v.short_name,
       v.long_description
FROM meta_network n
JOIN meta_vars v
  ON v.network_id = n.network_id
WHERE n.network_name IN :net_names
ORDER BY v.network_id;
""")

df = pd.read_sql(query, engine, params={"net_names": tuple(net_names)})
df

,vars_id,network_id,network_name,standard_name,net_var_name,display_name,short_name,long_description
0,671,5,BCH-GSO-HMP,soil_temperature,TSoil_Avg,Soil Temperature (Mean),soil_temperature_mean,Soil Temperature
1,569,5,BCH-GSO-HMP,air_temperature,Tx_Climatology,Temperature Climatology (Max.),air_temperaturet: maximum within days t: mean ...,Climatological mean of monthly mean maximum da...
2,572,5,BCH-GSO-HMP,lwe_thickness_of_precipitation_amount,Precip_Climatology,Precipitation Climatology,lwe_thickness_of_precipitation_amountt: sum wi...,Climatological mean of monthly total precipita...
3,463,5,BCH-GSO-HMP,air_temperature,MIN_TEMP,Temperature (Min.),air_temperature_minimum,Minimum daily temperature
4,468,5,BCH-GSO-HMP,surface_snow_thickness,SNOW_ON_THE_GROUND,Surface Snow Depth (Point),surface_snow_thickness_point,Amount of snow on the ground
5,467,5,BCH-GSO-HMP,thickness_of_snowfall_amount,ONE_DAY_SNOW,Snowfall Amount,thickness_of_snowfall_amount_sum,Daily snow accumulation
6,465,5,BCH-GSO-HMP,lwe_thickness_of_precipitation_amount,ONE_DAY_PRECIPITATION,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,Daily precipitation
7,570,5,BCH-GSO-HMP,air_temperature,Tn_Climatology,Temperature Climatology (Min.),air_temperaturet: minimum within days t: mean ...,Climatological mean of monthly mean minimum da...
8,464,5,BCH-GSO-HMP,air_temperature,MAX_TEMP,Temperature (Max.),air_temperature_maximum,Maximum daily temperature
9,466,5,BCH-GSO-HMP,thickness_of_rainfall_amount,ONE_DAY_RAIN,Rainfall Amount,thickness_of_rainfall_amount_sum,Daily rainfall


In [30]:
df_new_net_name

array(['BCH-SiteC', 'MVRD-AIR', 'BCH-GSO-HMP', 'MVRD', 'PC-FWx',
       'MVRD-WS', 'PRPA', 'BC-FERN', 'ENV ASP'], dtype=object)